In [2]:
import pandas as pd
import numpy as np


# LOAD FUTURES DATA


futures = pd.read_csv("KCc1 daily 1990-2026.csv")

futures["Exchange Date"] = pd.to_datetime(futures["Exchange Date"])

# Oldest first
futures = futures.sort_values("Exchange Date").reset_index(drop=True)

# Daily log returns (%)
futures["Return (%)"] = (
    np.log(futures["Close"] / futures["Close"].shift(1))
) * 100

futures["Previous Trading Date"] = futures["Exchange Date"].shift(1)


# LOAD WEATHER DATA


weather = pd.read_csv(
    "Brazil Minas Gerais 1990-2026.csv",
    skiprows=15
)

# Create proper dates
weather["Date"] = pd.to_datetime(
    weather["YEAR"].astype(str),
    format="%Y"
) + pd.to_timedelta(weather["DOY"] - 1, unit="D")


weather = weather[["Date", "T2M_MIN"]]


# MATCH WEATHER TO FUTURES


def interval_min_temp(row):

    if pd.isna(row["Previous Trading Date"]):
        return np.nan

    temps = weather.loc[
        (weather["Date"] > row["Previous Trading Date"]) &
        (weather["Date"] <= row["Exchange Date"]),
        "T2M_MIN"
    ]

    if len(temps) == 0:
        return np.nan

    return temps.min()


futures["Min Temp"] = futures.apply(interval_min_temp, axis=1)


# KEEP BRAZILIAN FROST SEASON


analysis = futures[
    futures["Exchange Date"].dt.month.isin([5, 6, 7, 8, 9])
].copy()

analysis = analysis.dropna(
    subset=["Return (%)", "Min Temp"]
)


# TEMPERATURE BUCKETS


bins = [-999, 0, 2, 5, 10, 999]

labels = [
    "≤0°C",
    "0–2°C",
    "2–5°C",
    "5–10°C",
    ">10°C"
]

analysis["Temperature Bucket"] = pd.cut(
    analysis["Min Temp"],
    bins=bins,
    labels=labels
)


# PANEL C TABLE


overall_mean = analysis["Return (%)"].mean()

analysis["Squared Dev"] = (
    analysis["Return (%)"] - overall_mean
) ** 2

total_variation = analysis["Squared Dev"].sum()

panel_c = (
    analysis
    .groupby("Temperature Bucket")
    .agg(
        Observations=("Return (%)", "count"),
        Mean_Return=("Return (%)", "mean"),
        Std_Dev=("Return (%)", "std"),
        Variation=("Squared Dev", "sum")
    )
)

panel_c["Observation Share (%)"] = (
    panel_c["Observations"] /
    panel_c["Observations"].sum()
) * 100

panel_c["Variation Share (%)"] = (
    panel_c["Variation"] /
    total_variation
) * 100

panel_c = panel_c.drop(columns="Variation")

print(panel_c.round(3))

# SAVE RESULTS


panel_c.to_csv("coffee_panel_c.csv")

analysis.to_csv(
    "coffee_frost_merged_daily.csv",
    index=False
)

print("\nDone!")
print("Created:")
print(" - coffee_panel_c.csv")
print(" - coffee_frost_merged_daily.csv")

                    Observations  Mean_Return  Std_Dev  Observation Share (%)  \
Temperature Bucket                                                              
≤0°C                           0          NaN      NaN                  0.000   
0–2°C                          3       21.005    3.795                  0.078   
2–5°C                         23        0.053    5.092                  0.597   
5–10°C                       758       -0.246    2.676                 19.688   
>10°C                       3066        0.016    2.304                 79.636   

                    Variation Share (%)  
Temperature Bucket                       
≤0°C                              0.000  
0–2°C                             5.725  
2–5°C                             2.411  
5–10°C                           23.072  
>10°C                            68.793  

Done!
Created:
 - coffee_panel_c.csv
 - coffee_frost_merged_daily.csv


/var/folders/zz/f4d8ydn15kqfjy20b3p_dqy80000gn/T/ipykernel_22601/1456452613.py:109: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("Temperature Bucket")


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path



# FILE PATHS


FUTURES_FILE = Path("KCc1 daily 1990-2026.csv")

WEATHER_FILES = {
    "Minas Gerais": Path("Brazil Minas Gerais 1990-2026.csv"),
    "Parana": Path("Brazil parana daily 1990-2026.csv"),
    "Sao Paulo": Path("Brazil Sao Paulo daily 1990-2026.csv"),
}


# LOAD FUTURES DATA


futures = pd.read_csv(FUTURES_FILE)

futures["Exchange Date"] = pd.to_datetime(
    futures["Exchange Date"],
    format="%d-%b-%Y",
    errors="coerce"
)

futures["Close"] = pd.to_numeric(
    futures["Close"],
    errors="coerce"
)

futures = (
    futures
    .dropna(subset=["Exchange Date", "Close"])
    .sort_values("Exchange Date")
    .drop_duplicates(subset=["Exchange Date"])
    .reset_index(drop=True)
)

# Daily continuously compounded percentage return
futures["Return (%)"] = (
    np.log(futures["Close"] / futures["Close"].shift(1)) * 100
)

futures["Previous Trading Date"] = (
    futures["Exchange Date"].shift(1)
)


# FUNCTION TO LOAD EACH NASA POWER WEATHER FILE


def load_nasa_weather(filepath, region_name):
    """
    Load one NASA POWER daily weather file.

    Assumes:
    - the first 15 rows contain metadata;
    - YEAR and DOY identify the date;
    - T2M_MIN is the daily minimum 2-metre temperature.
    """

    weather = pd.read_csv(
        filepath,
        skiprows=15
    )

    required_columns = {"YEAR", "DOY", "T2M_MIN"}

    missing_columns = required_columns.difference(weather.columns)

    if missing_columns:
        raise ValueError(
            f"{filepath} is missing these columns: "
            f"{sorted(missing_columns)}"
        )

    weather["YEAR"] = pd.to_numeric(
        weather["YEAR"],
        errors="coerce"
    )

    weather["DOY"] = pd.to_numeric(
        weather["DOY"],
        errors="coerce"
    )

    weather["T2M_MIN"] = pd.to_numeric(
        weather["T2M_MIN"],
        errors="coerce"
    )

    # NASA POWER sometimes uses values such as -999 for missing data
    weather.loc[
        weather["T2M_MIN"] <= -900,
        "T2M_MIN"
    ] = np.nan

    weather["Date"] = pd.to_datetime(
        weather["YEAR"].astype("Int64").astype(str)
        + weather["DOY"].astype("Int64").astype(str).str.zfill(3),
        format="%Y%j",
        errors="coerce"
    )

    temperature_column = f"{region_name} Min Temp"

    weather = weather.rename(
        columns={"T2M_MIN": temperature_column}
    )

    return (
        weather[["Date", temperature_column]]
        .dropna(subset=["Date"])
        .drop_duplicates(subset=["Date"])
        .sort_values("Date")
        .reset_index(drop=True)
    )



# LOAD AND COMBINE THE THREE WEATHER SERIES


weather_frames = []

for region, filepath in WEATHER_FILES.items():

    region_weather = load_nasa_weather(
        filepath=filepath,
        region_name=region
    )

    weather_frames.append(region_weather)


# Merge all state series by calendar date
weather = weather_frames[0]

for region_weather in weather_frames[1:]:

    weather = weather.merge(
        region_weather,
        on="Date",
        how="outer"
    )


state_temperature_columns = [
    "Minas Gerais Min Temp",
    "Parana Min Temp",
    "Sao Paulo Min Temp"
]

weather = weather.sort_values("Date").reset_index(drop=True)


# CREATE A COMBINED BRAZILIAN FROST VARIABLE


# Lowest temperature across the three states on each calendar day
weather["Brazil Regional Min Temp"] = weather[
    state_temperature_columns
].min(axis=1)


def coldest_region(row):
    """
    Return the region with the lowest temperature on that date.
    """

    temperatures = row[state_temperature_columns].dropna()

    if temperatures.empty:
        return np.nan

    coldest_column = temperatures.idxmin()

    return coldest_column.replace(" Min Temp", "")


weather["Coldest Region"] = weather.apply(
    coldest_region,
    axis=1
)



# MATCH WEATHER TO EACH FUTURES TRADING INTERVAL


def get_interval_weather(row):
    """
    For each trading date, use all weather observations after
    the previous trading date and up to the current trading date.

    For example, Monday's return uses Saturday, Sunday and Monday
    weather when Friday was the previous trading date.
    """

    previous_date = row["Previous Trading Date"]
    current_date = row["Exchange Date"]

    if pd.isna(previous_date) or pd.isna(current_date):
        return pd.Series({
            "Minas Gerais Interval Min": np.nan,
            "Parana Interval Min": np.nan,
            "Sao Paulo Interval Min": np.nan,
            "Brazil Interval Min": np.nan,
            "Coldest State in Interval": np.nan,
            "Coldest Weather Date": pd.NaT
        })

    interval = weather.loc[
        (weather["Date"] > previous_date)
        & (weather["Date"] <= current_date)
    ].copy()

    if interval.empty:
        return pd.Series({
            "Minas Gerais Interval Min": np.nan,
            "Parana Interval Min": np.nan,
            "Sao Paulo Interval Min": np.nan,
            "Brazil Interval Min": np.nan,
            "Coldest State in Interval": np.nan,
            "Coldest Weather Date": pd.NaT
        })

    regional_minimums = {
        "Minas Gerais Interval Min":
            interval["Minas Gerais Min Temp"].min(),

        "Parana Interval Min":
            interval["Parana Min Temp"].min(),

        "Sao Paulo Interval Min":
            interval["Sao Paulo Min Temp"].min()
    }

    # Locate the coldest region-date observation in the interval
    long_interval = interval.melt(
        id_vars="Date",
        value_vars=state_temperature_columns,
        var_name="Region",
        value_name="Temperature"
    ).dropna(subset=["Temperature"])

    if long_interval.empty:

        brazil_min = np.nan
        coldest_state = np.nan
        coldest_date = pd.NaT

    else:

        coldest_observation = long_interval.loc[
            long_interval["Temperature"].idxmin()
        ]

        brazil_min = coldest_observation["Temperature"]

        coldest_state = coldest_observation["Region"].replace(
            " Min Temp",
            ""
        )

        coldest_date = coldest_observation["Date"]

    return pd.Series({
        **regional_minimums,
        "Brazil Interval Min": brazil_min,
        "Coldest State in Interval": coldest_state,
        "Coldest Weather Date": coldest_date
    })


interval_weather = futures.apply(
    get_interval_weather,
    axis=1
)

futures = pd.concat(
    [futures, interval_weather],
    axis=1
)



# RESTRICT SAMPLE TO THE BRAZILIAN FROST SEASON


analysis = futures.loc[
    futures["Exchange Date"].dt.month.isin([5, 6, 7, 8, 9])
].copy()

analysis = analysis.dropna(
    subset=["Return (%)", "Brazil Interval Min"]
)

# TEMPERATURE BUCKETS


temperature_bins = [
    -np.inf,
    0,
    2,
    5,
    10,
    np.inf
]

temperature_labels = [
    r"<=0C",
    "0-2C",
    "2-5C",
    "5-10C",
    ">10C"
]

analysis["Temperature Bucket"] = pd.cut(
    analysis["Brazil Interval Min"],
    bins=temperature_bins,
    labels=temperature_labels,
    include_lowest=True,
    right=True
)



# CREATE THE NEW PANEL C TABLE


overall_mean = analysis["Return (%)"].mean()

analysis["Squared Deviation"] = (
    analysis["Return (%)"] - overall_mean
) ** 2

total_variation = analysis["Squared Deviation"].sum()


panel_c = (
    analysis
    .groupby(
        "Temperature Bucket",
        observed=False
    )
    .agg(
        Observations=("Return (%)", "count"),
        Mean_Return=("Return (%)", "mean"),
        Std_Dev=("Return (%)", "std"),
        Variation=("Squared Deviation", "sum")
    )
)

panel_c["Observation Share (%)"] = (
    panel_c["Observations"]
    / panel_c["Observations"].sum()
    * 100
)

panel_c["Variation Share (%)"] = (
    panel_c["Variation"]
    / total_variation
    * 100
)

panel_c = panel_c.drop(
    columns="Variation"
)



# STATE-LEVEL COMPARISON TABLES


def produce_temperature_table(data, temperature_column):
    """
    Produce the same descriptive table for one temperature measure.
    """

    sample = data.dropna(
        subset=["Return (%)", temperature_column]
    ).copy()

    sample["Bucket"] = pd.cut(
        sample[temperature_column],
        bins=temperature_bins,
        labels=temperature_labels,
        include_lowest=True,
        right=True
    )

    sample_mean = sample["Return (%)"].mean()

    sample["Squared Deviation"] = (
        sample["Return (%)"] - sample_mean
    ) ** 2

    total_squared_deviation = (
        sample["Squared Deviation"].sum()
    )

    table = (
        sample
        .groupby("Bucket", observed=False)
        .agg(
            Observations=("Return (%)", "count"),
            Mean_Return=("Return (%)", "mean"),
            Std_Dev=("Return (%)", "std"),
            Variation=("Squared Deviation", "sum")
        )
    )

    table["Observation Share (%)"] = (
        table["Observations"]
        / table["Observations"].sum()
        * 100
    )

    table["Variation Share (%)"] = (
        table["Variation"]
        / total_squared_deviation
        * 100
    )

    return table.drop(columns="Variation")


comparison_tables = {
    "Minas Gerais": produce_temperature_table(
        analysis,
        "Minas Gerais Interval Min"
    ),

    "Parana": produce_temperature_table(
        analysis,
        "Parana Interval Min"
    ),

    "Sao Paulo": produce_temperature_table(
        analysis,
        "Sao Paulo Interval Min"
    ),

    "Combined Brazil": produce_temperature_table(
        analysis,
        "Brazil Interval Min"
    )
}


# Compare the number of observations in each bucket
observation_comparison = pd.concat(
    {
        name: table["Observations"]
        for name, table in comparison_tables.items()
    },
    axis=1
)

observation_comparison.columns.name = "Temperature Measure"



# COLD-EVENT DIAGNOSTIC TABLE


cold_events = (
    analysis.loc[
        analysis["Brazil Interval Min"] <= 5,
        [
            "Exchange Date",
            "Previous Trading Date",
            "Return (%)",
            "Brazil Interval Min",
            "Coldest State in Interval",
            "Coldest Weather Date",
            "Minas Gerais Interval Min",
            "Parana Interval Min",
            "Sao Paulo Interval Min",
            "Temperature Bucket"
        ]
    ]
    .sort_values(
        ["Brazil Interval Min", "Exchange Date"]
    )
)



# DISPLAY RESULTS


print("\nCOMBINED THREE-STATE PANEL C TABLE")
print(panel_c.round(3))

print("\nOBSERVATION COUNTS BY TEMPERATURE MEASURE")
print(observation_comparison)

print("\nCOLDEST MATCHED EVENTS")
print(cold_events.head(30).to_string(index=False))


# SAVE RESULTS


weather.to_csv(
    "combined_three_state_weather.csv",
    index=False
)

analysis.to_csv(
    "coffee_three_state_merged_daily.csv",
    index=False
)

panel_c.to_csv(
    "coffee_three_state_panel_c.csv"
)

observation_comparison.to_csv(
    "temperature_measure_comparison.csv"
)

cold_events.to_csv(
    "coffee_cold_event_diagnostics.csv",
    index=False
)

for name, table in comparison_tables.items():

    safe_name = (
        name.lower()
        .replace(" ", "_")
    )

    table.to_csv(
        f"panel_c_{safe_name}.csv"
    )


print("\nFiles created:")
print(" - combined_three_state_weather.csv")
print(" - coffee_three_state_merged_daily.csv")
print(" - coffee_three_state_panel_c.csv")
print(" - temperature_measure_comparison.csv")
print(" - coffee_cold_event_diagnostics.csv")
print(" - panel_c_minas_gerais.csv")
print(" - panel_c_parana.csv")
print(" - panel_c_sao_paulo.csv")
print(" - panel_c_combined_brazil.csv")


COMBINED THREE-STATE PANEL C TABLE
                    Observations  Mean_Return  Std_Dev  Observation Share (%)  \
Temperature Bucket                                                              
<=0C                          12        4.246   10.659                  0.311   
0-2C                          26       -0.177    5.712                  0.674   
2-5C                         126       -0.618    2.958                  3.264   
5-10C                       1094       -0.083    2.487                 28.342   
>10C                        2602        0.024    2.294                 67.409   

                    Variation Share (%)  
Temperature Bucket                       
<=0C                              6.148  
0-2C                              3.420  
2-5C                              4.773  
5-10C                            28.334  
>10C                             57.326  

OBSERVATION COUNTS BY TEMPERATURE MEASURE
Temperature Measure  Minas Gerais  Parana  Sao Paulo  Combi

In [20]:
"""
Boudoukh et al. (2007) Table 2 analogue for Arabica coffee futures (KC front
month) and Brazilian frost, using a single frost threshold W* = 5C.

Model 1 (linear):    R_t = a + b*W_t + e_t
Model 2 (piecewise): R_t = a + b1*max(0, W* - W_t) + b2*max(0, W* - W_t)^2 + e_t

W_t = daily min of T2M_MIN across Minas Gerais, Sao Paulo, Parana.
Sample: Brazilian frost season (May-Sep). HAC (Newey-West, 5 lags) t-stats.
"""
import pandas as pd
import numpy as np
import statsmodels.api as sm

UP = "."
WSTAR = 5.0

#data
def read_power(path, name):
    df = pd.read_csv(path, skiprows=15)
    df["date"] = pd.to_datetime(df["YEAR"].astype(str), format="%Y") \
                 + pd.to_timedelta(df["DOY"] - 1, unit="D")
    df = df[["date", "T2M_MIN"]].rename(columns={"T2M_MIN": name})
    df.loc[df[name] <= -900, name] = np.nan
    return df

temp = (read_power(f"{UP}/Brazil Minas Gerais 1990-2026.csv", "MG")
        .merge(read_power(f"{UP}/Brazil parana daily 1990-2026.csv", "PR"), on="date")
        .merge(read_power(f"{UP}/Brazil sao paulo daily 1990-2026.csv", "SP"), on="date"))
temp["Wmin"] = temp[["MG", "PR", "SP"]].min(axis=1)
temp_idx = temp.set_index("date")["Wmin"]

kc = pd.read_csv(f"{UP}/KCc1 daily 1990-2026.csv")
kc.columns = ["date", "close"]
kc["date"] = pd.to_datetime(kc["date"], format="%d-%b-%Y")
kc = kc.sort_values("date").reset_index(drop=True)
kc["ret"] = 100 * np.log(kc["close"] / kc["close"].shift(1))
kc = kc.dropna(subset=["ret"]).copy()
kc["prev_date"] = kc["date"].shift(1)

def gap_min(row):
    if pd.isna(row["prev_date"]):
        return np.nan
    days = pd.date_range(row["prev_date"] + pd.Timedelta(days=1), row["date"])
    return temp_idx.reindex(days).min()

kc["Wt"] = kc.apply(gap_min, axis=1)
data = kc.dropna(subset=["Wt", "ret"]).copy()
winter = data[data["date"].dt.month.isin([5, 6, 7, 8, 9])].copy()

#  regressions 
y = winter["ret"].values
Wt = winter["Wt"].values
z = np.maximum(0.0, WSTAR - Wt)

m1 = sm.OLS(y, sm.add_constant(Wt)).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
m2 = sm.OLS(y, sm.add_constant(np.column_stack([z, z**2]))).fit(
    cov_type="HAC", cov_kwds={"maxlags": 5})
joint = m2.f_test("x1 = 0, x2 = 0")

# table 
def cell(coef, t):
    return f"{coef: .4f}\n({t: .2f})"

rows = {
    "a":        [cell(m1.params[0], m1.tvalues[0]), cell(m2.params[0], m2.tvalues[0])],
    "b":        [cell(m1.params[1], m1.tvalues[1]), ""],
    "b1":       ["", cell(m2.params[1], m2.tvalues[1])],
    "b2":       ["", cell(m2.params[2], m2.tvalues[2])],
    "R2 (%)":   [f"{m1.rsquared*100:.2f}", f"{m2.rsquared*100:.2f}"],
    "Adj R2 (%)": [f"{m1.rsquared_adj*100:.2f}", f"{m2.rsquared_adj*100:.2f}"],
    "Joint p (b1=b2=0)": ["", f"{float(joint.pvalue):.4f}"],
    "N":        [f"{len(winter)}", f"{len(winter)}"],
    "Freeze days (Wt < W*)": ["", f"{(z > 0).sum()}"],
}

print(f"Sample: May-Sep, {winter['date'].min().date()} to {winter['date'].max().date()}")
print(f"Frost threshold W* = {WSTAR}C. Newey-West (5 lags) t-stats in parentheses.\n")

col1, col2 = "Model 1 (linear)", "Model 2 (piecewise quad)"
w0, w1, w2 = 22, 20, 26
print(f"{'':<{w0}}{col1:>{w1}}{col2:>{w2}}")
print("-" * (w0 + w1 + w2))
for name, (c1, c2) in rows.items():
    l1 = c1.split("\n"); l2 = c2.split("\n")
    for i in range(max(len(l1), len(l2))):
        lab = name if i == 0 else ""
        v1 = l1[i] if i < len(l1) else ""
        v2 = l2[i] if i < len(l2) else ""
        print(f"{lab:<{w0}}{v1:>{w1}}{v2:>{w2}}")
print("-" * (w0 + w1 + w2))

# tidy CSV version of the table
pd.DataFrame({
    "param": ["a", "b", "b1", "b2", "R2_pct", "AdjR2_pct", "joint_p", "N", "freeze_days"],
    "model1_linear": [m1.params[0], m1.params[1], np.nan, np.nan,
                      m1.rsquared*100, m1.rsquared_adj*100, np.nan, len(winter), np.nan],
    "model1_tstat": [m1.tvalues[0], m1.tvalues[1], np.nan, np.nan,
                     np.nan, np.nan, np.nan, np.nan, np.nan],
    "model2_piecewise": [m2.params[0], np.nan, m2.params[1], m2.params[2],
                         m2.rsquared*100, m2.rsquared_adj*100, float(joint.pvalue),
                         len(winter), (z > 0).sum()],
    "model2_tstat": [m2.tvalues[0], np.nan, m2.tvalues[1], m2.tvalues[2],
                     np.nan, np.nan, np.nan, np.nan, np.nan],
}).to_csv("frost_table.csv", index=False)
print("\nSaved tidy version to frost_table.csv")

# ---------------- plot ----------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

grid = np.linspace(-3, 25, 300)
fit_lin = m1.params[0] + m1.params[1] * grid
zg = np.maximum(0.0, WSTAR - grid)
fit_quad = m2.params[0] + m2.params[1] * zg + m2.params[2] * zg**2

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(Wt, y, s=6, alpha=0.3, color="gray", label="Winter days (May-Sep)")
ax.plot(grid, fit_lin, color="darkorange", lw=2,
        label=f"Linear: $R^2$={m1.rsquared*100:.2f}%")
ax.plot(grid, fit_quad, color="crimson", lw=2,
        label=f"Piecewise quadratic ($W^*$={WSTAR:.0f}$\\degree$C): $R^2$={m2.rsquared*100:.2f}%")
ax.axvline(WSTAR, ls="--", color="steelblue", lw=1, label=f"$W^*$ = {WSTAR:.0f}$\\degree$C")
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Min temperature across MG / SP / PR ($\\degree$C)")
ax.set_ylabel("KC front-month log return (%)")
ax.set_title("Coffee futures returns vs Brazilian minimum temperature\nLinear vs piecewise-quadratic freeze model")
ax.legend()
ax.set_xlim(-3.5, 25)
plt.tight_layout()
plt.savefig("frost_returns_fit.pdf", format="pdf", bbox_inches="tight")
print("Saved plot to frost_returns_fit.pdf")

Sample: May-Sep, 1990-05-01 to 2026-07-10
Frost threshold W* = 5.0C. Newey-West (5 lags) t-stats in parentheses.

                          Model 1 (linear)  Model 2 (piecewise quad)
--------------------------------------------------------------------
a                                   0.0285                   -0.0120
                                   ( 0.13)                   (-0.32)
b                                  -0.0043                          
                                   (-0.22)                          
b1                                                           -1.1650
                                                             (-2.87)
b2                                                            0.3175
                                                             ( 2.60)
R2 (%)                                0.00                      1.53
Adj R2 (%)                           -0.02                      1.48
Joint p (b1=b2=0)                                         